## 26-3-12

In [ ]:
import numpy as np
from pysr import PySRRegressor
import torch
from fepysr import FePySR

model_default = PySRRegressor(
    populations=8,
    population_size=50,
    ncycles_per_iteration=500,
    niterations=2000,  # Run forever
    early_stop_condition=(
        "stop_if(loss, complexity) = loss < 1e-9 && complexity < 40"
    ),
    timeout_in_seconds=60 * 60 * 24,
    maxsize=25,
    maxdepth=10,
    binary_operators=["*", "+", "-"],
    unary_operators=[
        "cos",
        "sqrt",
        # "log",
        "exp",
        "sin",
        "inv(x) = 1/x",
        # ^ Custom operator (julia syntax)
    ],
    extra_sympy_mappings={"inv": lambda x: 1 / x},
    elementwise_loss="loss(prediction, target) = (prediction - target)^2",
    # constraints={
    #     "/": (-1, 9),
    #     "square": 9,
    #     "cube": 9,
    #     "exp": 9,
    # },
    nested_constraints={
        "exp": {"exp": 0},  # 禁止 exp(exp(x)) 这样的嵌套
        # "inv": {"inv": 0},  # 禁止 inv(inv(x)) 这样的嵌套
    },
    # nested_constraints={
    #     "square": {"square": 1, "cube": 1, "exp": 0},
    #     "cube": {"square": 1, "cube": 1, "exp": 0},
    #     "exp": {"square": 1, "cube": 1, "exp": 0},
    # },
    # complexity_of_operators={"/": 2, "exp": 3},
    # complexity_of_constants=2,
    # select_k_features=4,
    # progress=True,
    # weight_randomize=0.1,
    # cluster_manager=None,
    # precision=64,
    # warm_start=True,
    # turbo=True
)

X = np.random.randn(200, 2)
y = np.sin(X[:, 0]) + X[:, 1]**2

# X=torch.randn(10,dtype=torch.float64)
# y=X*X

# z=fit(x,y,custom_pysr_model=model_default)
# z
# model = FePySR(overrides=["FMN.batch_size=50", "Parallel.num_workers=8",'Parallel.num_experiments=16','FMN.FMN_only=True'],custom_pysr_model=model_default)

model = FePySR(
    overrides=[
        "FMN.batch_size=50",
        "FMN.lr=0.5",
        "FMN.num_epochs=100",
        "FMN.loss=fepysr.optimization.squared_loss",
        "FMN.loss_parameter.lambda1=0.08",
        "FMN.loss_parameter.lambda2=0.001",
        "FMN.net.fun_list=null",
        "FMN.net.net_depth=4",
        "FMN.net.fun_re=4",
        "FMN.net.full_net=null",
        "FMN.device=auto",
        "FMN.FMN_only=False",
		
        "data_symbol.raw_data=null",
        "data_symbol.output_symbols=null",
        "data_symbol.fea_num=10",
        "data_symbol.pysr_num=6",
		
        "Parallel.num_experiments=16",
        "Parallel.num_workers=8",
		
        "pysr_params.populations=8",
        "pysr_params.population_size=50",
        "pysr_params.ncycles_per_iteration=500",
        "pysr_params.niterations=2000",
        "pysr_params.timeout_in_seconds=86400",
        "pysr_params.maxsize=25",
        "pysr_params.maxdepth=10",
        "pysr_params.early_stop_condition='stop_if(loss, complexity) = loss < 1e-9 && complexity < 40'",
        
        "pysr_params.binary_operators=['*','+','-']",
        "pysr_params.unary_operators=['cos','sqrt','exp','sin','inv(x) = 1/x']",
        "pysr_params.elementwise_loss='loss(prediction, target) = (prediction - target)^2'",
        "pysr_params.nested_constraints.exp.exp=0"
    ]
)

model.fit(X, y)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
--- Starting Feature Mapping Network ---
Set number of runs: 16, Allocated number of workers: 8
Mode: Parallel computation


d:\Scoop\Apps\miniconda3\26.1.1-1\envs\FePySR\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
d:\Scoop\Apps\miniconda3\26.1.1-1\envs\FePySR\lib\site-packages\juliacall\__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(
d:\Scoop\Apps\miniconda3\26.1.1-1\envs\FePySR\lib\site-packages\juliacall\__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(
d:\Scoop\Apps\miniconda3\26.1.1-1\envs\FePySR\lib\site-packages\juliacall\__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault

───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           2.238e+00  0.000e+00  y = 0.97123
2           1.957e+00  1.343e-01  y = sqrt(x₃)
3           4.516e-01  1.466e+00  y = x₁ * x₁
5           3.064e-01  1.939e-01  y = (x₁ * x₁) + x₀
6           1.767e-14  3.048e+01  y = (x₁ * x₁) + sin(x₀)
───────────────────────────────────────────────────────────────────────────────────────────────────


  - outputs\20260316_104213_HFr49E\hall_of_fame.csv


In [9]:
a=model.data_analyzer.features
type(a),a

(torch.Tensor,
 tensor([[-9.4829e-01, -5.2589e-01],
         [-6.3493e-02,  1.8550e-01],
         [ 2.1567e-01, -8.8468e-01],
         [-6.0812e-01,  2.7864e-01],
         [ 6.8646e-02,  6.8914e-01],
         [-1.2287e+00,  4.7385e-01],
         [-8.2891e-02,  2.1088e+00],
         [-8.9860e-01, -8.9863e-01],
         [-2.0780e+00,  4.3302e-01],
         [-5.4794e-01,  1.6166e-01],
         [ 1.2989e+00,  5.8424e-01],
         [-7.4707e-02, -3.7232e-01],
         [ 7.3461e-01,  2.2098e-01],
         [ 6.6660e-01,  1.4327e+00],
         [-9.1832e-01, -3.9706e-01],
         [-6.4937e-01, -4.0842e-01],
         [ 3.1052e-01, -1.0708e+00],
         [ 9.5215e-01, -1.2595e+00],
         [-1.1559e+00, -5.3986e-01],
         [-9.6190e-01,  2.0349e+00],
         [ 7.4386e-02, -4.6454e-01],
         [ 1.3730e+00,  2.8170e-01],
         [ 6.9253e-01, -1.8315e-01],
         [ 3.5259e-01,  3.4735e-02],
         [-3.9697e-01, -3.8649e-01],
         [ 8.4365e-01, -8.2879e-01],
         [ 1.0337e-01, 

In [ ]:
import numpy as np
from pysr import PySRRegressor
import torch
from utils.fepysr import FePySR

data=np.load("fun_rec_data/fun_rec_1.npy")
x,y=torch.from_numpy(data[:,:1]),torch.from_numpy(data[:,1:])
model = FePySR(overrides=["Parallel.num_workers=8",'Parallel.num_experiments=32'])
model.fit(x, y)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
--- Starting Feature Mapping Network ---
Set number of runs: 32, Allocated number of workers: 8
Mode: Parallel computation


d:\Scoop\Apps\miniconda3\26.1.1-1\envs\FePySR\lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...
d:\Scoop\Apps\miniconda3\26.1.1-1\envs\FePySR\lib\site-packages\juliacall\__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(
d:\Scoop\Apps\miniconda3\26.1.1-1\envs\FePySR\lib\site-packages\juliacall\__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault. To avoid this, import juliacall before importing torch. For updates, see https://github.com/pytorch/pytorch/issues/78829.
  warnings.warn(
d:\Scoop\Apps\miniconda3\26.1.1-1\envs\FePySR\lib\site-packages\juliacall\__init__.py:61: UserWarning: torch was imported before juliacall. This may cause a segfault

───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
4           2.667e+04  0.000e+00  y = exp(sin(inv(x₅)))
5           2.219e+04  1.816e-01  y = (x₃ * x₀) + x₁
9           2.179e+04  4.017e-03  y = ((x₃ * x₀) + (x₁ - -0.25301)) + x₃
10          1.890e+01  7.048e+00  y = x₁ + ((x₁ + sqrt(1.7707)) * (x₀ + x₃))
11          4.941e+00  1.342e+00  y = (x₃ * x₀) + (x₁ + (x₁ * (x₀ + x₃)))
13          2.807e+00  2.828e-01  y = (x₃ * x₀) + (((x₅ + x₁) * (x₀ + x₃)) + x₁)
14          5.403e-01  1.648e+00  y = ((x₃ * x₀) + x₁) + ((x₁ + sqrt(1.7707)) * (x₀ + x₃))
15          1.959e-01  1.014e+00  y = (((x₁ + 0.93417) * (x₃ + x₀)) - 0.3546) + ((x₀ * x₃) +...
                                       x₁)
17          3.334e-11  1.125e+01  y = ((((x₁ + 1) * (x₃ + x₀)) - 0.33403) + ((x₀ * x₃) + x₁)...
                                      ) + 0.33403
───────────────────────────────────────────────────────────────────

In [11]:
modell.best_equation_

'X0*(X0 + 1)*(X0^4 + X0^2 + 1)'

In [13]:
print(modell.feature_names_)

['X0', 'X0**4', 'exp(X0)', 'cos(X0)', 'sin(X0)', 'X0**2', 'cos(cos(X0))', 'sin(cos(X0))', '2*X0**2', 'exp(cos(X0))', 'cos(cos(X0))**2']


In [ ]:
import numpy as np
from pysr import PySRRegressor
import torch
from utils.fepysr import FePySR

data=np.load("fun_rec_data/fun_rec_1.npy")
x,y=torch.from_numpy(data[:,:1]),torch.from_numpy(data[:,1:])
modell = FePySR(overrides=["Parallel.num_workers=8",'Parallel.num_experiments=32'])
modell.fit(x, y)

ModuleNotFoundError: No module named 'fepysr'

In [5]:
modell.all_feature_names_

[('X0**4', 198),
 ('exp(X0)', 179),
 ('cos(X0)', 157),
 ('sin(X0)', 153),
 ('X0**2', 143),
 ('cos(cos(X0))', 126),
 ('sin(cos(X0))', 125),
 ('2*X0**2', 75),
 ('exp(cos(X0))', 71),
 ('cos(cos(X0))**2', 71)]

In [7]:
print(modell.solved_pysr_model)

PySRRegressor.equations_ = [
	   pick      score                       equation          loss  complexity
	0         0.000000                       sqrt(x1)  2.631391e+04           2
	1         0.307767                        x1 + x1  1.934301e+04           3
	2         1.979492                  x1 * sqrt(x1)  2.672032e+03           4
	3         2.227887                 x1 * (x5 + x0)  2.879272e+02           5
	4         2.032543          (x5 + x0) * (x1 + x5)  4.941268e+00           7
	5  >>>>  12.521559  ((x1 + 1.0) + x5) * (x5 + x0)  6.572803e-11           9
]
